In [ ]:

!git clone https://github.com/ThanhTrunggDEV/GenAI.git
%cd GenAI
!ls -F

In [ ]:
# Cài đặt dependencies
!pip install -q diffusers transformers accelerate bitsandbytes peft
!pip install -q packaging>=23.2.0,<26.0.0 fastcore==1.8.0
!pip install -q -r requirements.txt

In [ ]:
# Set PYTHONPATH để python tìm thấy module motif
import os
os.environ['PYTHONPATH'] = os.getcwd()

# Chuẩn bị dữ liệu training
!python -m motif.data.prepare

# Trích xuất đặc trưng (Stage 1)
!python -m motif.models.visual_encoder
!python -m motif.models.cultural_encoder
!python -m motif.models.combine_embeddings

In [ ]:
# Training Configuration for LoRA (Low-Rank Adaptation)
# Batch size 4 * Accumulation 4 = Effective Batch Size 16

!accelerate launch --num_processes=1 --mixed_precision="fp16" train_diffusion.py \
    --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
    --output_dir="outputs/hmong-pattern-lora" \
    --num_train_epochs=30 \
    --train_batch_size=4 \
    --gradient_accumulation_steps=4 \
    --mixed_precision="fp16"

In [ ]:
!python -m motif.pipeline.generate --prompt "Hmong spiral pattern in indigo" --checkpoint "outputs/hmong-pattern-lora"

In [ ]:
!python -m motif.pipeline.evaluate --generated "outputs/generated"

In [ ]:
!zip -r outputs.zip outputs/